# Aspect-Based Sentiment Analysis for Velox Foods — Local Pipeline (Ollama)

This notebook adapts the "free option" from Marcus's brief: run a **local, open-source LLM
via Ollama** over our 200 customer reviews, and extract structured sentiment for an overall
rating plus four business-relevant aspects: **Food, Service, Price, Ambiance**.

This costs nothing but electricity. The trade-off we're testing: is a small local model
smart enough, and fast enough, to do this job at scale?


---
## 1.&nbsp; Set-up 🛠️

**Note:** You must have the Ollama app running on your computer, and you must have pulled
a model first, e.g. by running `ollama run llama3.2` in your terminal, before running this code.


In [1]:
import ollama
import pandas as pd
import time
from pydantic import BaseModel, Field
import json


Quick connection test to make sure Python can talk to the local model.

In [2]:
try:
    response = ollama.chat(
        model='llama3.2',
        messages=[
            {
                'role': 'user',
                'content': 'Are you ready to analyse some restaurant reviews? Answer yes or no.'
            }
        ]
    )
    print("Model Response:", response['message']['content'])
except Exception as e:
    print("Error:", e)
    print("⚠️ Make sure the Ollama app is running and you have pulled llama3.2!")


Model Response: Yes.


### 1.1 Why `ollama.chat` and not `ollama.generate`?

We use `ollama.chat` because:

1. **It's stateless by default** — we build a brand-new `messages` list for every review, so the
   model carries zero memory between rows. No "memory tax", no risk of one review's content
   leaking into another's analysis.
2. **Built-in instruction formatting** — modern instruct-tuned models expect a strict separation
   between system rules and user content. `ollama.chat` handles this formatting for us, which in
   practice produces far more reliable JSON than gluing one big string together ourselves.

### 1.2 Enforcing structure with Pydantic

Instead of hoping the model outputs valid JSON, we **define the exact shape we want** with a
Pydantic schema and pass it to Ollama. This guarantees the response will match our schema —
no more guessing whether the JSON parses.


In [3]:
# Define the structure for a single aspect (optional — may not be mentioned)
class AspectScore(BaseModel):
    score: float | None = None   # -1.0 to +1.0, or null if not mentioned
    quote: str | None = None     # short supporting evidence from the review

# overall_rating is REQUIRED — every review expresses some overall impression,
# even if individual aspects (food/service/price/ambiance) are not mentioned.
# Note there is no default value here and no `| None`, so Pydantic will raise a
# ValidationError if the model tries to omit it or return null for the score.
class OverallRating(BaseModel):
    score: float = Field(..., description="Overall sentiment score, -1.0 to +1.0. Required, cannot be null.")
    quote: str | None = None

# Define the overall structure we expect the model to return for each review
class ReviewAnalysis(BaseModel):
    overall_rating: OverallRating          # required — no default, must always be present
    food: AspectScore = AspectScore()
    service: AspectScore = AspectScore()
    price: AspectScore = AspectScore()
    ambiance: AspectScore = AspectScore()


LLMs are unpredictable. Even with structured output, a model might drop a key instead of
returning `"score": null`. Setting defaults on `food`, `service`, `price` and `ambiance`
means Pydantic quietly fills in `None` instead of crashing our pipeline.

**`overall_rating` is different on purpose.** We do *not* give it a default and we do *not*
allow `None` for its score. Every review expresses some overall impression of the visit, even
a one-line review — so if the model fails to provide it, that's a real failure we want to know
about, not something to silently paper over. This means `ReviewAnalysis.model_validate_json(...)`
will raise a `ValidationError` if `overall_rating` (or its score) is missing. We'll catch that
in our `parse_sentiment` function below and treat it as a failed review needing a retry, rather
than quietly recording a blank overall score.


---
## 2.&nbsp; Prompt Engineering 📝

We are Data Analysts, not poets — we need structured output, not a vague paragraph. The
**system prompt** sets the rules of the game. We explicitly tell the model it's a restaurant
review analyst, define our four aspects, and inject our Pydantic schema directly into the
prompt so the model knows the exact JSON shape we expect.


In [4]:
# We dynamically inject the schema into the prompt
schema_instructions = ReviewAnalysis.model_json_schema()

SYSTEM_PROMPT = f"""
You are an expert restaurant industry analyst working for Velox Foods, a fast-food chain.
Your task is to read a single customer review and extract:

1. overall_rating: the customer's OVERALL sentiment about the visit as a whole.
   This field is REQUIRED and must NEVER be null. Every review expresses some overall
   impression, even a short one — if the customer didn't spell it out explicitly, infer
   it from the tone and content of the review as a whole. The score must always be a
   number, never null.

2. Sentiment scores for four specific business aspects, each OPTIONAL (use null if the
   review does not mention that aspect at all):
   - food: taste, freshness, temperature, portion size, accuracy of the order
   - service: staff behaviour, speed, friendliness, order accuracy at the counter/drive-thru
   - price: whether the customer felt the price was fair for what they received
   - ambiance: cleanliness, seating, noise, atmosphere of the restaurant itself

For every score (overall_rating and the four aspects), use this scale:
-1.0 (Strong Negative)
-0.5 (Negative)
 0.0 (Neutral / Mixed)
+0.5 (Positive)
+1.0 (Strong Positive)
null (Not Mentioned) — ONLY valid for food, service, price, ambiance. NEVER valid for overall_rating.

Also extract a short quote (a few words, copied from the review) as evidence for each
non-null score. If an aspect is not mentioned at all, both its score and quote must be null.

Be strict: only assign a score to food/service/price/ambiance if the review text actually
discusses it. Do not infer or assume — if the customer didn't mention service, service must
be null. overall_rating is the only exception to this rule, since it is required.

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(schema_instructions, indent=2)}
"""


---
## 3.&nbsp; Analysing a Single Review 🧪

Before running this on 200 rows, let's test it on one deliberately mixed review — positive
on food, negative on service. This is exactly the kind of nuance a star rating alone can't capture.


In [5]:
test_review = (
    "The burger itself was actually really good, hot and fresh with a great patty. "
    "But the girl at the counter was so rude, she rolled her eyes when I asked for ketchup. "
    "Took forever to get my order too. Won't be rushing back because of how I was treated."
)


### 3.1 Setting the temperature

* **High temperature (e.g. 0.8):** good for creative writing, the model takes risks.
* **Low temperature (e.g. 0):** good for data extraction — deterministic and predictable,
  which is exactly what we want for a repeatable analysis pipeline.


In [6]:
response = ollama.chat(
    model='llama3.2',
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': test_review}
    ],
    # This passes the Pydantic schema to Ollama to enforce structure
    format=ReviewAnalysis.model_json_schema(),
    options={'temperature': 0}
)


Let's look at the raw response object — it carries a lot of useful technical metadata.

In [7]:
print(response.model_dump_json(indent=4))

{
    "model": "llama3.2",
    "created_at": "2026-06-25T22:45:38.751959Z",
    "done": true,
    "done_reason": "stop",
    "total_duration": 6092243542,
    "load_duration": 132239792,
    "prompt_eval_count": 965,
    "prompt_eval_duration": 2353452000,
    "eval_count": 154,
    "eval_duration": 3602806000,
    "message": {
        "role": "assistant",
        "content": "{\n  \"overall_rating\": {\n    \"score\": -1.0,\n    \"quote\": \"Won't be rushing back because of how I was treated.\"\n  },\n  \"food\": {\n    \"score\": 1.0,\n    \"quote\": \"the burger itself was actually really good, hot and fresh with a great patty.\"\n  },\n  \"service\": {\n    \"score\": -1.0,\n    \"quote\": \" Took forever to get my order too. She rolled her eyes when I asked for ketchup.\"\n  },\n  \"price\": {\n    \"score\": null,\n    \"quote\": null\n  },\n  \"ambiance\": {\n    \"score\": null,\n    \"quote\": null\n  }\n}",
        "thinking": null,
        "images": null,
        "tool_name":

### 3.2 Understanding the metadata

* **`total_duration`**: time taken in nanoseconds. With 200 reviews, this tells us whether
  the "free" option is actually usable at scale, or painfully slow.
* **`eval_count`**: number of output tokens. This is the kind of thing API providers (like Groq)
  bill you for — useful to track now so we have a fair point of comparison later.
* **`message`**: the container holding the actual analysis content.


In [8]:
print(response['message']['content'])

{
  "overall_rating": {
    "score": -1.0,
    "quote": "Won't be rushing back because of how I was treated."
  },
  "food": {
    "score": 1.0,
    "quote": "the burger itself was actually really good, hot and fresh with a great patty."
  },
  "service": {
    "score": -1.0,
    "quote": " Took forever to get my order too. She rolled her eyes when I asked for ketchup."
  },
  "price": {
    "score": null,
    "quote": null
  },
  "ambiance": {
    "score": null,
    "quote": null
  }
}


### 3.3 From string to dictionary

The model returns a **string** that looks like JSON, but Python still treats it as plain text
until we parse it. We validate it straight into our Pydantic model, then flatten it for pandas.


In [9]:
# Convert the string to a validated Pydantic object
validated_data = ReviewAnalysis.model_validate_json(response['message']['content'])

# Convert the Pydantic object into a dictionary for Pandas
data = validated_data.model_dump()
data


{'overall_rating': {'score': -1.0,
  'quote': "Won't be rushing back because of how I was treated."},
 'food': {'score': 1.0,
  'quote': 'the burger itself was actually really good, hot and fresh with a great patty.'},
 'service': {'score': -1.0,
  'quote': ' Took forever to get my order too. She rolled her eyes when I asked for ketchup.'},
 'price': {'score': None, 'quote': None},
 'ambiance': {'score': None, 'quote': None}}

In [10]:
# Flatten nested keys (e.g. food -> food_score, food_quote; overall_rating -> overall_rating_score, ...)
pd.json_normalize(data, sep='_')


,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
0,-1.0,Won't be rushing back because of how I was tre...,1.0,"the burger itself was actually really good, ho...",-1.0,Took forever to get my order too. She rolled ...,None,None,None,None


---
## 4.&nbsp; Processing a DataFrame 🧱

Now we scale the single-review logic up into reusable functions: one to call the model,
one to parse the sentiment JSON, one to parse the technical metadata, and one main loop
that ties it all together.


**1. The analysis function** — wraps the Ollama call with retries so one bad response doesn't kill the whole run.

In [11]:
def analyse_review_local(review_text, review_id, retries=3):
    """
    Gets the raw ollama response. Retries on failure.

    Because overall_rating is required and cannot be null, we also treat a
    response that FAILS our schema validation (e.g. the model returned
    overall_rating.score as null anyway) as a reason to retry — not just
    network/API errors. Models occasionally ignore "required" instructions,
    so we give it a few more chances before giving up.

    Returns None only after all retries fail.
    """
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model='llama3.2',
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': review_text}
                ],
                format=ReviewAnalysis.model_json_schema(),
                options={'temperature': 0}
            )

            # Convert the ChatResponse object to a standard dictionary
            response_dict = response.model_dump() if hasattr(response, 'model_dump') else dict(response)

            # Validate now, not just later in parse_sentiment, so a violation of
            # our "overall_rating is required" rule actually triggers a retry.
            content_str = response_dict.get('message', {}).get('content', '{}')
            ReviewAnalysis.model_validate_json(content_str)  # raises if invalid

            return response_dict

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for review {review_id}: {e}")
            if attempt < retries - 1:
                time.sleep(1)
            else:
                print(f"All {retries} attempts failed for review {review_id}.")
                return None


**2. The metadata parser** — extracts duration/token counts for our later cost & speed comparison.

In [12]:
class OllamaMetadata(BaseModel):
    model_name: str | None = Field(default=None, alias="model")
    created_at: str | None = None
    total_duration_ns: int | None = Field(default=None, alias="total_duration")
    load_duration_ns: int | None = Field(default=None, alias="load_duration")
    prompt_eval_duration_ns: int | None = Field(default=None, alias="prompt_eval_duration")
    generation_duration_ns: int | None = Field(default=None, alias="eval_duration")
    input_token_count: int | None = Field(default=None, alias="prompt_eval_count")
    output_token_count: int | None = Field(default=None, alias="eval_count")


def parse_metadata(response, review_id):
    """
    Extracts technical metadata using Pydantic.
    Returns a structure with None values if keys are missing.
    """
    fallback_data = OllamaMetadata().model_dump()
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        validated_meta = OllamaMetadata.model_validate(response)
        final_dict = validated_meta.model_dump()
        final_dict["review_id"] = review_id
        return final_dict
    except Exception as e:
        print(f"Error parsing metadata for {review_id}: {e}")
        return fallback_data


**3. The sentiment parser** — turns the raw JSON string into a clean dict, falling back gracefully if parsing fails.

In [13]:
def parse_sentiment(response, review_id):
    """
    Extracts the aspect scores using Pydantic validation.
    Returns a consistent structure with None values if parsing fails — except
    overall_rating, which uses a sentinel score of None ONLY in this fallback
    path, to flag that the row needs manual review (it failed validation, it
    wasn't genuinely "not mentioned").
    """
    fallback_data = {
        "review_id": review_id,
        "overall_rating": {"score": None, "quote": None},
        "food": {"score": None, "quote": None},
        "service": {"score": None, "quote": None},
        "price": {"score": None, "quote": None},
        "ambiance": {"score": None, "quote": None},
    }

    if response is None:
        return fallback_data

    try:
        message = response.get('message', {})
        content_str = message.get('content', '{}')

        # This will raise a ValidationError if overall_rating is missing or null —
        # that's intentional, it's our required field.
        validated_data = ReviewAnalysis.model_validate_json(content_str)
        final_dict = validated_data.model_dump()
        final_dict["review_id"] = review_id
        return final_dict

    except Exception as e:
        print(f"Error parsing sentiment for review {review_id}: {e}")
        return fallback_data


**4. The main loop** — ties it all together, looping over every row in our DataFrame.

In [14]:
def process_reviews(df, text_col='text', id_col='review_id'):
    sentiment_records = []
    metadata_records = []

    total = len(df)
    for i, (index, row) in enumerate(df.iterrows(), start=1):
        r_id = row[id_col]
        text = row[text_col]

        raw_response = analyse_review_local(text, r_id)

        sentiment_data = parse_sentiment(raw_response, r_id)
        meta_data = parse_metadata(raw_response, r_id)

        sentiment_records.append(sentiment_data)
        metadata_records.append(meta_data)

        if i % 10 == 0 or i == total:
            print(f"Processed {i}/{total} reviews")

    df_aspects = pd.json_normalize(sentiment_records, sep='_')
    if not df_aspects.empty:
        df_aspects = df_aspects.set_index('review_id')

    df_metadata = pd.DataFrame(metadata_records)
    if not df_metadata.empty:
        df_metadata = df_metadata.set_index('review_id')

    return df_aspects, df_metadata


### 4.1 Running on a small dummy set

Before touching the real data, let's sanity-check the pipeline on a handful of made-up
restaurant reviews that deliberately mix positive and negative aspects.


In [15]:
dummy_data = [
    {
        "review_id": "dummy_1",
        "text": "Food was hot and delicious, best burger I've had in ages. Staff were a bit slow though."
    },
    {
        "review_id": "dummy_2",
        "text": "Way too expensive for what you get. Fries were cold and the dining area was filthy."
    },
    {
        "review_id": "dummy_3",
        "text": "Great value meal deal, and the new manager has clearly trained the team well - super friendly."
    },
    {
        "review_id": "dummy_4",
        "text": "Just grabbed a coffee, nothing special to report either way."
    },
    {
        "review_id": "dummy_5",
        "text": "Lovely cosy seating area, but my order was wrong twice and the chicken was dry."
    },
]

df_dummy = pd.DataFrame(dummy_data)
df_dummy


,review_id,text
0,dummy_1,"Food was hot and delicious, best burger I've h..."
1,dummy_2,Way too expensive for what you get. Fries were...
2,dummy_3,"Great value meal deal, and the new manager has..."
3,dummy_4,"Just grabbed a coffee, nothing special to repo..."
4,dummy_5,"Lovely cosy seating area, but my order was wro..."


In [16]:
dummy_aspects, dummy_metadata = process_reviews(df_dummy)


Processed 5/5 reviews


In [17]:
dummy_aspects

,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
review_id,,,,,,,,,,
dummy_1,-0.5,best burger I've had in ages.,1.0,hot and delicious,-0.5,NaN,NaN,NaN,NaN,NaN
dummy_2,-1.0,Way too expensive for what you get.,-1.0,Fries were cold,NaN,NaN,-1.0,Way too expensive for what you get.,-1.0,the dining area was filthy
dummy_3,1.0,super friendly,NaN,NaN,1.0,super friendly,NaN,NaN,NaN,NaN
dummy_4,-0.5,nothing special,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dummy_5,-1.0,my order was wrong twice and the chicken was dry.,-1.0,the chicken was dry,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
dummy_metadata

,model_name,created_at,total_duration_ns,load_duration_ns,prompt_eval_duration_ns,generation_duration_ns,input_token_count,output_token_count
review_id,,,,,,,,
dummy_1,llama3.2,2026-06-25T22:45:41.845874Z,3016210041,120980500,100076000,2793000000,928,119
dummy_2,llama3.2,2026-06-25T22:45:45.27651Z,3427715125,98869000,99781000,3226873000,926,135
dummy_3,llama3.2,2026-06-25T22:45:48.098102Z,2817310667,98188042,104082000,2612922000,926,110
dummy_4,llama3.2,2026-06-25T22:45:50.790952Z,2689842417,96639833,114113000,2476900000,919,105
dummy_5,llama3.2,2026-06-25T22:45:53.791715Z,2996090875,96394834,95934000,2801787000,925,120


If the scores and quotes above look sensible (e.g. `dummy_2` should show negative price,
negative food, negative ambiance, an overall_rating that is clearly negative too, and
service close to null since staff aren't mentioned), the pipeline logic is working. Tweak
the `SYSTEM_PROMPT` above and re-run this section if anything looks off before moving on to
the real data. Also double-check that `overall_rating` is never null across all five dummy
rows — that's the constraint we're relying on.


---
## 5.&nbsp; Running on the Real Velox Foods Dataset 🚀

Now we run this on the actual 200 customer reviews Marcus gave us.

**The plan:**
1. Load `reviews.csv`.
2. Test on a small sample (10 rows) first — running 200 rows locally can be slow, and we want
   to catch prompt issues before burning 200 rows' worth of time.
3. Once happy, scale up to the full 200 rows.
4. Save both the aspect scores and the metadata (timing/tokens) to CSV — we need the metadata
   for the cost-benefit report.
5. **Time the whole run.** This number directly feeds into the "is local fast enough" answer
   we owe Marcus.


In [19]:
df = pd.read_csv('restaurant_reviews_sample.csv')
print(df.shape)
df.head()


(200, 5)


,review_id,user_id,business_id,stars,text
0,5dIleZpTxKwjRqBercHV1w,6G7JMGMjwsni6UM-jgn6JA,iSRTaT9WngzB8JJ2YKJUig,5.0,Great meal and service. I didn't realize how l...
1,LETLlxzUzv4eHkD84jea-Q,jOLRUxLpI0NgLsuWHNq9rA,iSRTaT9WngzB8JJ2YKJUig,5.0,Please don't listen to anyone Who gives this p...
2,9sXneDy_Oyd-n-bCUz6TLg,lnl9XpsQ01k-pnxnf-0DAA,iSRTaT9WngzB8JJ2YKJUig,3.0,Seems like a must while in NO. Mostly I enjoye...
3,0tBSNwWA3eMtbBC2PgzQtQ,FJpsfNM620MiQVtGXsznpQ,iSRTaT9WngzB8JJ2YKJUig,1.0,"One of the worst meals I have ever eaten, I wa..."
4,YMiytO01Ocd6rT-0XWfacQ,bK0PskbRUT7Eo6FYOjydlw,iSRTaT9WngzB8JJ2YKJUig,2.0,I had been wanting to try this place for years...


In [20]:
# Step 1: small sample first
df_sample = df.sample(10, random_state=42).reset_index(drop=True)
df_sample


,review_id,user_id,business_id,stars,text
0,HDAgNEU6DcjgRyMlEJKnLQ,lZL5gFg6nNY8GN-pwfWzkg,iSRTaT9WngzB8JJ2YKJUig,5.0,A group of 6 of us were visiting had we heard ...
1,fYpVtfrZpuF0kJYwte-GgQ,jfpqJpSwQnL-6eblk7hIhQ,iSRTaT9WngzB8JJ2YKJUig,5.0,"The service is move move move, but good, good,..."
2,2k009P4y2KEqemzpcbqETw,8eVOX9evLuBx9yCDP3Qr_w,iSRTaT9WngzB8JJ2YKJUig,3.0,"Cafeteria style, nothing fancy but the turnip ..."
3,FzNGQVF8rtvo6dXpaeGHWg,VD-n1LtJQhLOyG2ll_s_mw,iSRTaT9WngzB8JJ2YKJUig,3.0,Checked out this place for breakfast because i...
4,Z6ffijbxOKUixHImgPiVPQ,GdXRnK65bFmrhV4OOGx0hA,iSRTaT9WngzB8JJ2YKJUig,5.0,"As a native, I must say....THE BEST po boy in ..."
5,jxh4fKN8H1ekLcozHaVpFg,bhy4xEpbPjivoI0bcuoZ3A,iSRTaT9WngzB8JJ2YKJUig,1.0,Tasted like poop! It really did. Don't recomme...
6,SrbPGR41yrP1-ElahE-3Fw,LBvk8_yBMTql4hFhKZ5NSA,iSRTaT9WngzB8JJ2YKJUig,5.0,Its Mother's not fancy but great food whenever...
7,4xjgcRJEcS9AsPUnh5pZCQ,412qT2gXxzhFi59oLOmuIw,iSRTaT9WngzB8JJ2YKJUig,2.0,Looking forward to local restaurant. There wer...
8,mz1uW2E6sFiNvaRa9V4kwA,sng9pz8HRxH9hkIBoujVXw,iSRTaT9WngzB8JJ2YKJUig,1.0,In town visiting for the first time and read t...
9,tJ9htfCsiyvk8p2zngXspQ,93s5XlhSBaIBcCpPecXu4w,iSRTaT9WngzB8JJ2YKJUig,3.0,"It was alright. There is a ton of ""hype"" on th..."


In [21]:
sample_start = time.time()
sample_aspects, sample_metadata = process_reviews(df_sample)
sample_elapsed = time.time() - sample_start
print(f"Sample of {len(df_sample)} reviews took {sample_elapsed:.1f} seconds "
      f"({sample_elapsed/len(df_sample):.2f}s per review)")


Processed 10/10 reviews
Sample of 10 reviews took 33.3 seconds (3.33s per review)


In [22]:
sample_aspects

,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
review_id,,,,,,,,,,
HDAgNEU6DcjgRyMlEJKnLQ,1.0,Would go back again.,1.0,Everyone enjoyed their meals.,1.0,Good service,1.0,Good value for the amount of food,NaN,NaN
fYpVtfrZpuF0kJYwte-GgQ,0.5,good,1.0,"well not really, but there was a lot. And the ...",1.0,"move move move, but good",-1.0,This food is expensive for a counter service r...,NaN,NaN
2k009P4y2KEqemzpcbqETw,-0.5,nothing fancy,1.0,delicious with big hunks of ham,NaN,NaN,NaN,NaN,NaN,NaN
FzNGQVF8rtvo6dXpaeGHWg,-0.5,I wouldn't go out of my way,0.5,the ham was good but I could probably make bet...,NaN,NaN,NaN,NaN,NaN,NaN
Z6ffijbxOKUixHImgPiVPQ,1.0,THE BEST po boy in NOLA,1.0,Ferdi Special and File gumbo are best menu items.,NaN,NaN,NaN,NaN,NaN,NaN
jxh4fKN8H1ekLcozHaVpFg,-1.0,poop,-1.0,tasted like poop,NaN,NaN,NaN,NaN,NaN,NaN
SrbPGR41yrP1-ElahE-3Fw,-0.5,not fancy,1.0,great food,NaN,NaN,NaN,NaN,NaN,NaN
4xjgcRJEcS9AsPUnh5pZCQ,-1.0,We ate and as leaving advised guy by the door ...,-1.0,"Fries, gumbo everything else cold.",-1.0,She took off and left us thinking she would be...,NaN,NaN,-1.0,Place is pretty dirty too
mz1uW2E6sFiNvaRa9V4kwA,-2.0,"Lesson learned, 3 stars says a lot.",NaN,NaN,-1.0,"a simple ""pardon me"" would have sufficed",NaN,NaN,NaN,NaN


In [23]:
sample_metadata

,model_name,created_at,total_duration_ns,load_duration_ns,prompt_eval_duration_ns,generation_duration_ns,input_token_count,output_token_count
review_id,,,,,,,,
HDAgNEU6DcjgRyMlEJKnLQ,llama3.2,2026-06-25T22:45:57.150532Z,3301409000,120269333,178070000,3000973000,963,129
fYpVtfrZpuF0kJYwte-GgQ,llama3.2,2026-06-25T22:46:01.02055Z,3865673375,95228417,332489000,3435211000,1011,147
2k009P4y2KEqemzpcbqETw,llama3.2,2026-06-25T22:46:03.992474Z,2968431458,96998000,174750000,2694440000,954,116
FzNGQVF8rtvo6dXpaeGHWg,llama3.2,2026-06-25T22:46:07.356918Z,3359835417,95808167,330889000,2930380000,1002,126
Z6ffijbxOKUixHImgPiVPQ,llama3.2,2026-06-25T22:46:10.622116Z,3260820416,108739750,257060999,2892226000,992,125
jxh4fKN8H1ekLcozHaVpFg,llama3.2,2026-06-25T22:46:13.402366Z,2775908584,95798125,97550000,2580518000,928,112
SrbPGR41yrP1-ElahE-3Fw,llama3.2,2026-06-25T22:46:16.149097Z,2742406709,96780459,98080000,2545682000,932,110
4xjgcRJEcS9AsPUnh5pZCQ,llama3.2,2026-06-25T22:46:20.378038Z,4224587208,95774833,576165000,3549960000,1114,152
mz1uW2E6sFiNvaRa9V4kwA,llama3.2,2026-06-25T22:46:24.205033Z,3822245417,96665458,741214000,2981275000,1190,127


**Check the sample output carefully before scaling up:**
- Do the scores match how you'd read each review yourself?
- Are quotes actually pulled from the review text, not invented?
- Is anything being scored that wasn't actually mentioned (the model "hallucinating" an aspect)?

If something looks wrong, revise `SYSTEM_PROMPT` in section 2 and re-run sections 2-5 above
before scaling up. Iterating here is much cheaper than discovering a bad prompt after running
all 200 rows.


### 5.1 Scaling up to all 200 reviews

Once you're happy with the sample, run the full dataset. This will take a while locally —
that's exactly the data point we need for the report.


In [24]:
full_start = time.time()
ollama_aspects, ollama_metadata = process_reviews(df)
full_elapsed = time.time() - full_start

print(f"Full run of {len(df)} reviews took {full_elapsed:.1f} seconds "
      f"({full_elapsed/len(df):.2f}s per review on average)")


Processed 10/200 reviews
Processed 20/200 reviews
Processed 30/200 reviews
Processed 40/200 reviews
Processed 50/200 reviews
Processed 60/200 reviews
Processed 70/200 reviews
Processed 80/200 reviews
Processed 90/200 reviews
Processed 100/200 reviews
Processed 110/200 reviews
Processed 120/200 reviews
Processed 130/200 reviews
Processed 140/200 reviews
Processed 150/200 reviews
Processed 160/200 reviews
Processed 170/200 reviews
Processed 180/200 reviews
Processed 190/200 reviews
Processed 200/200 reviews
Full run of 200 reviews took 955.5 seconds (4.78s per review on average)


In [25]:
ollama_aspects.head(10)

,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
review_id,,,,,,,,,,
5dIleZpTxKwjRqBercHV1w,0.5,hence the rating,1.0,the food was very authentic and flavorful,1.0,NaN,NaN,NaN,NaN,NaN
LETLlxzUzv4eHkD84jea-Q,4.8,The ordering system is different but effective.,1.0,the biscuits are great (fluffy and delicious),NaN,NaN,NaN,NaN,NaN,NaN
9sXneDy_Oyd-n-bCUz6TLg,-0.5,Late breakfast,-1.0,debris (shredded roast beef in juice) on top o...,NaN,NaN,NaN,NaN,NaN,NaN
0tBSNwWA3eMtbBC2PgzQtQ,-1.0,One of the worst meals I have ever eaten,-1.0,Cold dry chicken greens had no flavor cooked i...,NaN,NaN,NaN,NaN,-1.0,The place was dirty
YMiytO01Ocd6rT-0XWfacQ,-1.0,I just don't get it.,-1.0,the bread was soggy and fell apart,-0.5,NaN,NaN,NaN,NaN,NaN
vbM-THkWu7o2KbFj0u38ug,-0.5,nothing special,-1.0,not spectacular,NaN,NaN,NaN,NaN,NaN,NaN
vYxz_9dlsvTvtxscaQzE2w,1.0,must!,1.0,try the fried shrimp!,1.0,NaN,NaN,NaN,NaN,NaN
khuy3aKkXP7TQzDEezNCYw,1.0,Awesome,1.0,Perfect,1.0,super efficient,NaN,NaN,NaN,NaN
eC2RdN3SF6JDw8Ki-yP5fA,-0.5,LOL,1.0,Pancakes were awesome and grits were fantastical!,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
ollama_metadata.head(10)

,model_name,created_at,total_duration_ns,load_duration_ns,prompt_eval_duration_ns,generation_duration_ns,input_token_count,output_token_count
review_id,,,,,,,,
5dIleZpTxKwjRqBercHV1w,llama3.2,2026-06-25T22:46:30.213198Z,3073474416,115305125,171235000,2784785000,942,120
LETLlxzUzv4eHkD84jea-Q,llama3.2,2026-06-25T22:46:33.310647Z,3093044833,96911208,170902000,2823207000,963,122
9sXneDy_Oyd-n-bCUz6TLg,llama3.2,2026-06-25T22:46:36.583233Z,3268293000,96245542,254421000,2915008000,969,126
0tBSNwWA3eMtbBC2PgzQtQ,llama3.2,2026-06-25T22:46:40.146397Z,3558043625,103400500,324086000,3127705000,1005,135
YMiytO01Ocd6rT-0XWfacQ,llama3.2,2026-06-25T22:46:43.508934Z,3358600583,101996750,406681000,2847145000,1041,123
vbM-THkWu7o2KbFj0u38ug,llama3.2,2026-06-25T22:46:46.414045Z,2900895000,98869375,255326000,2544169000,987,110
vYxz_9dlsvTvtxscaQzE2w,llama3.2,2026-06-25T22:46:49.405921Z,2987610042,100398459,250763000,2633703000,969,114
khuy3aKkXP7TQzDEezNCYw,llama3.2,2026-06-25T22:46:52.379026Z,2968983834,104492084,257283000,2604524000,992,113
eC2RdN3SF6JDw8Ki-yP5fA,llama3.2,2026-06-25T22:46:55.419502Z,3036362083,102443250,178279000,2753537000,965,119


### 5.2 Saving our work

We save three things:
1. The aspect scores (our actual ABSA output).
2. The technical metadata (for the speed/cost comparison).
3. A tiny summary file recording total run time and average tokens — we'll want this exact
   number again when writing the final report to Marcus.


In [27]:
import os
os.makedirs('../data', exist_ok=True)

ollama_aspects.to_csv('../data/ollama_aspects.csv')
ollama_metadata.to_csv('../data/ollama_metadata.csv')

summary = {
    "pipeline": "ollama_local",
    "model": "llama3.2",
    "num_reviews": len(df),
    "total_seconds": full_elapsed,
    "avg_seconds_per_review": full_elapsed / len(df),
    "avg_output_tokens": ollama_metadata['output_token_count'].mean(),
    "cost_usd": 0.0,  # local model — electricity only
}

with open('../data/ollama_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

summary


{'pipeline': 'ollama_local',
 'model': 'llama3.2',
 'num_reviews': 200,
 'total_seconds': 955.5288581848145,
 'avg_seconds_per_review': 4.777644290924072,
 'avg_output_tokens': np.float64(128.245),
 'cost_usd': 0.0}

In [28]:
import os
print(os.path.abspath('../data'))
print(os.listdir('../data'))

/Users/stetsenko/data
['ollama_metadata.csv', 'ollama_aspects.csv', 'ollama_summary.json']


---
## Key Takeaways 🧠

* **Strict instructions matter.** Without a schema and explicit rules ("don't infer aspects
  that aren't mentioned"), the model will happily guess — which would poison our comparison
  with Groq later.
* **Structured output (`format=...`) turns an LLM into a data pipeline component**, not just
  a chatbot — this is what makes ABSA at scale possible at all.
* **Speed is the real question mark for "free".** Note your `full_elapsed` and
  `avg_seconds_per_review` numbers above. If Velox Foods eventually wants to process
  thousands of reviews a week across 514 stores, multiply that average out — does "free"
  still look free once you account for how long it ties up your machine?
* Keep `ollama_aspects.csv`, `ollama_metadata.csv` and `ollama_summary.json` safe — the next
  notebook (Groq API pipeline) will be compared directly against these numbers.
